In [ ]:
# Cell 1

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -o book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# Cell 2

# calculate ratings
user_counts = df_ratings['user'].value_counts()
isbn_counts = df_ratings['isbn'].value_counts()

# remove users with < 200 ratings and books with < 100 ratings
df_ratings_rm = df_ratings[~df_ratings['user'].isin(user_counts[user_counts < 200].index)]
df_ratings_rm = df_ratings_rm[~df_ratings_rm['isbn'].isin(isbn_counts[isbn_counts < 100].index)]


df = pd.merge(df_ratings_rm, df_books, on='isbn')

df = df.drop_duplicates(['title', 'user'])

df_pivot = df.pivot(index='title', columns='user', values='rating').fillna(0)

matrix = csr_matrix(df_pivot.values)

In [ ]:
# Cell 3

model_knn = NearestNeighbors(metric='cosine', algorithm='brute')

model_knn.fit(matrix)

In [ ]:
# Cell 4

def get_recommends(book=""):
    book_row = df_pivot.loc[book].values.reshape(1, -1)

    distances, indices = model_knn.kneighbors(book_row, n_neighbors=6)

    recs = []
    for i in range(5, 0, -1):
        title = df_pivot.index[indices[0][i]]
        distance = distances[0][i]
        recs.append([title, distance])

    return [book, recs]

In [ ]:
#Cell 5

books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False

  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]

  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()